# Quality Assessment
Now it's your turn. Follow the steps on the platform and use what you've learnt to see how reliable the data is.

In [1]:
import pandas as pd

Load our cleaned DataFrames

In [38]:
# orders_cl.csv
url = "https://drive.google.com/file/d/1QiWLCP_60mpXEYS6A1YEH3S0Whg288xy/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders_cl = pd.read_csv(path)

# orderlines_cl.csv
url = "https://drive.google.com/file/d/1hu2Kh3rAOQVC0gGLQN8Zs_HMboVXlu2B/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines_cl = pd.read_csv(path)

# products_cl.csv
url = "https://drive.google.com/file/d/1IPnKYLSnrST0HBZMeSeNrhFfWU31QvwB/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_cl = pd.read_csv(path)

## 1.&nbsp; Define Pandas display format

In [39]:
# your code here
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_rows', 1000)

## 2.&nbsp; Exclude unwanted orders

In [40]:
# save the dataframes
orders_df = orders_cl.copy()
orderlines_df = orderlines_cl.copy()
products_df = products_cl.copy()

In [41]:
# save the dataframes
orders_df["state"].value_counts()

state
Shopping Basket    117809
Completed           46605
Place Order         40883
Pending             14374
Cancelled            7233
Name: count, dtype: int64

In [42]:
# keep only orders completed, pending or placed
orders_df.info()
orders_df["state"].unique()
orders_df.query("state in ['Completed', 'Place Order', 'Pending']", inplace=True)
orders_df.info()
orders_df["state"].unique()

<class 'pandas.DataFrame'>
RangeIndex: 226904 entries, 0 to 226903
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226904 non-null  int64  
 1   created_date  226904 non-null  str    
 2   total_paid    226904 non-null  float64
 3   state         226904 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 13.7 MB
<class 'pandas.DataFrame'>
Index: 101862 entries, 1 to 226903
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      101862 non-null  int64  
 1   created_date  101862 non-null  str    
 2   total_paid    101862 non-null  float64
 3   state         101862 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 6.7 MB


<ArrowStringArray>
['Completed', 'Pending', 'Place Order']
Length: 3, dtype: str

In [46]:
# Filtering orderlines based on id
orderlines_df = orderlines_df[orderlines_df["id_order"].isin(orders_df["order_id"])]
orderlines_df.info()

id_order
428186    31
323269    17
319417    14
333300    13
520342    13
          ..
527394     1
527396     1
527384     1
527397     1
527401     1
Name: count, Length: 78190, dtype: int64

In [44]:
# Filtering orders based on id
orders_df = orders_df[orders_df["order_id"].isin(orderlines_df["id_order"])]
orders_df.info()

<class 'pandas.DataFrame'>
Index: 79846 entries, 1 to 226903
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      79846 non-null  int64  
 1   created_date  79846 non-null  str    
 2   total_paid    79846 non-null  float64
 3   state         79846 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 5.2 MB


## 3.&nbsp; Exclude orders with unknown products


In [45]:
# Remove unknow products based on "Sku"
orderlines_df = orderlines_df.loc[
    orderlines_df["sku"].isin(products_df["sku"])
]
orderlines_df.info()

<class 'pandas.DataFrame'>
Index: 98664 entries, 6 to 216249
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                98664 non-null  int64  
 1   id_order          98664 non-null  int64  
 2   product_id        98664 non-null  int64  
 3   product_quantity  98664 non-null  int64  
 4   sku               98664 non-null  str    
 5   unit_price        98664 non-null  float64
 6   date              98664 non-null  str    
dtypes: float64(1), int64(4), str(2)
memory usage: 8.5 MB


In [9]:
# Remove unknown products in orders based on "id"
orders_df = orders_df.loc[
    orders_df["order_id"].isin(orderlines_df["id_order"])
]
orders_df.info()

<class 'pandas.DataFrame'>
Index: 78190 entries, 1 to 226903
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      78190 non-null  int64  
 1   created_date  78190 non-null  str    
 2   total_paid    78190 non-null  float64
 3   state         78190 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 5.1 MB


## 4.&nbsp; Explore the revenue from different tables

#### Step 1:
Create the `unit_price_total` as `orderlines.unit_price` * `orderlines.product_quantity`

In [10]:
# get unit_price total per order
orderlines_df.info()
orderlines_df["unit_price_total"] = orderlines_df["product_quantity"] * orderlines_df["unit_price"]
orderlines_df.head(10)

<class 'pandas.DataFrame'>
Index: 98664 entries, 6 to 216249
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                98664 non-null  int64  
 1   id_order          98664 non-null  int64  
 2   product_id        98664 non-null  int64  
 3   product_quantity  98664 non-null  int64  
 4   sku               98664 non-null  str    
 5   unit_price        98664 non-null  float64
 6   date              98664 non-null  str    
dtypes: float64(1), int64(4), str(2)
memory usage: 8.5 MB


,id,id_order,product_id,product_quantity,sku,unit_price,date,unit_price_total
6,1119116,299545,0,1,OWC0100,47.49,2017-01-01 01:46:16,47.49
7,1119119,299546,0,1,IOT0014,18.99,2017-01-01 01:50:34,18.99
8,1119120,295347,0,1,APP0700,72.19,2017-01-01 01:54:11,72.19
14,1119133,299555,0,1,BEA0065,256.49,2017-01-01 02:18:45,256.49
15,1119134,299556,0,1,CRU0039-A,60.90,2017-01-01 02:20:14,60.90
21,1119145,299561,0,1,PEB0015,142.49,2017-01-01 02:38:50,142.49
23,1119154,299563,0,1,BEA0065,256.49,2017-01-01 02:42:05,256.49
24,1119155,299564,0,1,SAT0010,18.99,2017-01-01 02:43:37,18.99
30,1119181,299571,0,1,SYN0139,166.24,2017-01-01 04:00:44,166.24
37,1119188,299578,0,1,LOG0191,142.49,2017-01-01 08:21:26,142.49


#### Step 2:
Group by `id_order`, summarising by the sum of `unit_price_total`

In [11]:
# make a df with the sum of units
order_totals_df = orderlines_df.groupby("id_order")[["unit_price_total"]].sum()
order_totals_df = order_totals_df.reset_index()
order_totals_df.head()

,id_order,unit_price_total
0,241423,129.16
1,242832,10.77
2,243330,77.99
3,245275,149.00
4,245595,105.98


In [12]:
# merge with orders (with inner I want to keep only values that are in both tables)
orders_df = orders_df.merge(order_totals_df,
               left_on = "order_id",
               right_on = "id_order",
               how = "inner")
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 78190 entries, 0 to 78189
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          78190 non-null  int64  
 1   created_date      78190 non-null  str    
 2   total_paid        78190 non-null  float64
 3   state             78190 non-null  str    
 4   id_order          78190 non-null  int64  
 5   unit_price_total  78190 non-null  float64
dtypes: float64(2), int64(2), str(2)
memory usage: 5.7 MB


### What is the average difference between `total_paid` and `unit_price_total`?

In [27]:
# your code here
# orders_df["total_paid"].mean() - orders_df["unit_price_total"].mean()
products_df.head()

,sku,name,desc,price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,1,1364


### What is the distribution of these differences?

In [14]:
# your code here

### Can all the differences be explained by shipping costs? If not, what are other plausible explanations?

In [15]:
# your explanation here

### If there are differences that you can’t explain: what should you do with these orders?

In [16]:
# your code here

## 5.&nbsp; Become confident about your dataset

Let's hear your thoughts about what you discovered.

In [17]:
# your thoughts & code here

Do not forget to download your quality controlled DataFrames

In [18]:
from google.colab import files

#orders_quality_controlled.to_csv("orders_qu.csv", index=False)
#files.download("orders_qu.csv")

#orderlines_quality_controlled.to_csv("orderlines_qu.csv", index=False)
#files.download("orderlines_qu.csv")

ModuleNotFoundError: No module named 'google.colab'